# QANet Deep Learning Mechanisms — Error Analysis

**COMP5329 Deep Learning — Assignment 1, 2026**

This notebook systematically identifies and documents **14 critical errors** in the deep learning mechanisms that cause:
- Incorrect gradient flow
- Unstable optimization
- Poor initialization
- Broken regularization
- Flawed attention computation
- Broken residual connections
- Wrong learning rate scheduling
- Broken momentum accumulation
- Incorrect normalization
- Broken span prediction

---

## Error 1: Xavier Uniform Initialization — Wrong Formula

### 📍 Location
`Models/Initializations/xavier.py`, lines 33–36

### ❌ Current (Incorrect) Code
```python
def xavier_uniform_(tensor: torch.Tensor, gain: float = 1.0) -> torch.Tensor:
    fan_in, fan_out = _calculate_fan(tensor)
    std = gain * math.sqrt(2.0 / (fan_in * fan_out))  # WRONG: multiply instead of add
    bound = math.sqrt(3.0) * std
```

### 🐛 The Problem
- Uses product `(fan_in * fan_out)` in the denominator instead of sum
- Example: if `fan_in = 300, fan_out = 96`:
  - **Wrong**: `std = sqrt(2 / 28800) ≈ 0.0083` (too small!)
  - **Correct**: `std = sqrt(2 / 396) ≈ 0.071` (7× larger)
- Weights initialized too close to zero → **vanishing gradients and slow convergence**

### ✅ Correct Formula (Glorot/Xavier Uniform)
$$\text{std} = \text{gain} \cdot \sqrt{\frac{2}{\text{fan\_in} + \text{fan\_out}}}$$

$$\text{bound} = \sqrt{3} \cdot \text{std}$$

### 🔧 Fix
Replace the denominator:
```python
std = gain * math.sqrt(2.0 / (fan_in + fan_out))  # Add sum, not product
```

---

## Error 2: Kaiming Uniform Initialization — Missing Factor 2

### 📍 Location
`Models/Initializations/kaiming.py`, lines 30–36

### ❌ Current (Incorrect) Code
```python
def kaiming_uniform_(tensor: torch.Tensor, mode: str = "fan_in") -> torch.Tensor:
    fan_in, fan_out = _calculate_fan(tensor)
    fan = fan_in if mode == "fan_in" else fan_out
    std = math.sqrt(1.0 / fan)  # WRONG: missing the factor 2
    bound = math.sqrt(3.0) * std
```

### 🐛 The Problem
- He/Kaiming initialization is specifically designed for ReLU activation
- Missing the factor of 2 in the numerator
- Example: if `fan_in = 300`:
  - **Wrong**: `std = sqrt(1 / 300) ≈ 0.0577`
  - **Correct**: `std = sqrt(2 / 300) ≈ 0.0816` (1.41× larger)
- Leads to **dead ReLU problem**: many neurons stuck at 0

### ✅ Correct Formula (He Initialization for ReLU)
$$\text{std} = \sqrt{\frac{2}{\text{fan}}}$$

$$\text{bound} = \sqrt{3} \cdot \text{std}$$

### 🔧 Fix
```python
std = math.sqrt(2.0 / fan)  # Add the missing factor 2
```

---

## Error 3: StepLR Scheduler — Exponential Decay is Additive!

### 📍 Location
`Schedulers/step_scheduler.py`, lines 24–28

### ❌ Current (Incorrect) Code
```python
def get_lr(self):
    t = self.last_epoch
    return [
        base_lr * self.gamma * (t // self.step_size)  # WRONG: multiplication
        for base_lr in self.base_lrs
    ]
```

### 🐛 The Problem
- Uses linear multiplication: `lr = base_lr * 0.5 * (t // step_size)`
- **At step 0**: `lr = base_lr * 0.5 * 0 = 0` ⚠️ **Learning rate is zero!**
- **At step 5000** (step_size): `lr = base_lr * 0.5 * 1 = 0.5 * base_lr`
- **At step 10000**: `lr = base_lr * 0.5 * 2 = 1.0 * base_lr`

This is completely broken! Learning rate becomes zero immediately.

### ✅ Correct Formula (Exponential Step Decay)
$$\text{lr}_t = \text{base\_lr} \cdot \gamma^{\lfloor t / \text{step\_size} \rfloor}$$

With γ=0.5:
- **At step 0**: `lr = base_lr * 0.5^0 = base_lr` ✓
- **At step 5000**: `lr = base_lr * 0.5^1 = 0.5 * base_lr` ✓
- **At step 10000**: `lr = base_lr * 0.5^2 = 0.25 * base_lr` ✓

### 🔧 Fix
Use exponentiation instead of multiplication:
```python
def get_lr(self):
    t = self.last_epoch
    return [
        base_lr * (self.gamma ** (t // self.step_size))  # Exponential decay
        for base_lr in self.base_lrs
    ]
```

### 🚨 **CRITICAL SEVERITY**
This error completely breaks gradient-based optimization!

---

## Error 4: Cosine Annealing Scheduler — Missing 0.5 Factor

### 📍 Location
`Schedulers/cosine_scheduler.py`, lines 22–26

### ❌ Current (Incorrect) Code
```python
def get_lr(self):
    t = self.last_epoch
    return [
        self.eta_min + (base_lr - self.eta_min) * (1 + math.cos(math.pi * t / self.T_max))
        for base_lr in self.base_lrs
    ]
```

### 🐛 The Problem
- The cosine function ranges from -1 to +1
- So `(1 + cos(x))` ranges from 0 to 2
- Missing the 0.5 factor to normalize to [0, 1]

Trajectory (wrong):
- **At t=0**: `cos(0) = 1` → `lr = eta_min + (base_lr - eta_min) * 2 = base_lr + (base_lr - eta_min)` (too high!)
- **At t=T_max/2**: `cos(π/2) = 0` → `lr = eta_min + (base_lr - eta_min) * 1 = base_lr` (wrong!)
- **At t=T_max**: `cos(π) = -1` → `lr = eta_min + (base_lr - eta_min) * 0 = eta_min` ✓

### ✅ Correct Formula (Cosine Annealing)
$$\text{lr}_t = \eta_{\min} + \frac{1}{2}(\text{base\_lr} - \eta_{\min})\left(1 + \cos\left(\frac{\pi t}{T_{\max}}\right)\right)$$

With the 0.5 factor:
- **At t=0**: `lr = eta_min + 0.5 * (base_lr - eta_min) * 2 = base_lr` ✓
- **At t=T_max/2**: `lr = eta_min + 0.5 * (base_lr - eta_min) * 1 = 0.5 * (base_lr + eta_min)` ✓
- **At t=T_max**: `lr = eta_min` ✓

### 🔧 Fix
```python
def get_lr(self):
    t = self.last_epoch
    return [
        self.eta_min + 0.5 * (base_lr - self.eta_min) * (1 + math.cos(math.pi * t / self.T_max))
        for base_lr in self.base_lrs
    ]
```

---

## Error 5: Adam Optimizer — Wrong Sign for Weight Decay

### 📍 Location
`Optimizers/adam.py`, lines 53–54

### ❌ Current (Incorrect) Code
```python
# Weight decay
if wd != 0.0:
    grad = grad.add(p, alpha=-wd)  # WRONG: negative sign!
```

### 🐛 The Problem
- L2 regularization should **penalize** large weights
- Correct: `grad_new = grad + wd * w`
- Current: `grad_new = grad - wd * w` (subtracts the weight!)

**This has the OPPOSITE effect**: encourages weights to grow instead of shrink!

### ✅ Correct Formula (L2 Regularization)
$$\mathcal{L}_{\text{new}} = \mathcal{L} + \lambda \|w\|_2^2$$
$$\frac{\partial \mathcal{L}_{\text{new}}}{\partial w} = \frac{\partial \mathcal{L}}{\partial w} + 2\lambda w$$

So we should **add** the weight term to the gradient.

### 🔧 Fix
```python
# Weight decay (L2 regularization)
if wd != 0.0:
    grad = grad.add(p, alpha=wd)  # Positive alpha
```

---

## Error 6: Multi-Head Attention — Missing Scaling Factor

### 📍 Location
`Models/encoder.py`, lines 62–64

### ❌ Current (Incorrect) Code
```python
attn = torch.bmm(q, k.transpose(1, 2))  # No scaling!
attn = mask_logits(attn, attn_mask)
attn = F.softmax(attn, dim=2)
```

### 🐛 The Problem
- Attention scores are raw dot products between query and key vectors
- In QANet: `d_k = d_model / num_heads = 96 / 8 = 12`
- Dot product has magnitude scaling with `sqrt(d_k)`
- Without scaling by `1/sqrt(d_k)`, attention logits are **very large**

**Consequences:**
- Large logits → softmax becomes extremely peaked (near one-hot)
- Gradient through softmax: `∂softmax/∂x = softmax(x) * (1 - softmax(x))`
- When softmax ≈ [0.99, 0.01, ...], gradients vanish
- Attention weights don't update properly

### ✅ Correct Formula (Scaled Dot-Product Attention)
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

The scaling ensures attention logits have variance ≈ 1.

### 🔧 Fix
```python
attn = torch.bmm(q, k.transpose(1, 2)) * self.scale  # Apply 1/sqrt(d_k)
attn = mask_logits(attn, attn_mask)
attn = F.softmax(attn, dim=2)
```

Note: `self.scale = 1.0 / math.sqrt(self.d_k)` is already computed in `__init__`.

---

## Error 7: EncoderBlock — Broken Residual and Duplicate Conv

### 📍 Location
`Models/encoder.py`, lines 94–111

### ❌ Current (Incorrect) Code
```python
def forward(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    out = self.pos(x)                  # Add positional encoding
    res = out                          # Save as residual
    out = self.normb(out)              # Layer norm

    for i, conv in enumerate(self.convs):
        out = conv(out)
        out = self.norms[i](out)
        out = conv(out)                # ❌ ERROR 7a: Conv applied TWICE!
        out = self.act(out)
        out = out + res
        if (i + 1) % 2 == 0:
            out = self.conv_drops[i](out)

    out = self.self_att(out, mask)
    out = res                          # ❌ ERROR 7b: Overwrites with OLD residual!
    out = self.drop(out)
    # ...
```

### 🐛 The Problems

**Problem 7a: Duplicate Convolution**
- Line 99: `out = conv(out)` applied twice
- Should apply conv once, then activation
- Current flow: `conv → conv → act` (wrong)
- Correct flow: `conv → norm → act` (depthwise separable pattern)

**Problem 7b: Discarding Self-Attention Output**
- Line 111: `out = res` **completely discards** the self-attention computation
- `out` just computed self-attention: crucial for long-range dependencies
- Then line 111 overwrites it with `res` (the pre-attention residual)
- **Consequence**: All self-attention gradients are dead; model can't learn attention**
- All 8 attention heads are wasted computation!

### ✅ Correct Flow
1. Apply convolutions with residuals
2. Apply self-attention
3. **Add** self-attention output to residual (don't replace)
4. Apply feed-forward with residual

### 🔧 Fix
```python
def forward(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    out = self.pos(x)
    res = out
    out = self.normb(out)

    for i, conv in enumerate(self.convs):
        out = conv(out)
        out = self.norms[i](out)
        out = self.act(out)           # Remove duplicate conv, add activation
        out = out + res
        if (i + 1) % 2 == 0:
            out = self.conv_drops[i](out)

    out = self.self_att(out, mask)
    out = out + res                   # Add residual, don't overwrite!
    out = self.drop(out)

    res = out
    out = self.norme(out)
    out = self.fc(out.transpose(1, 2)).transpose(1, 2)
    out = self.act(out)
    out = out + res
    out = self.drop(out)
    return out
```

### 🚨 **CRITICAL SEVERITY**
This breaks the core architecture! Self-attention gradients don't flow.

---


## Error 9: Lambda Scheduler — Addition Instead of Multiplication

### 📍 Location
`Schedulers/lambda_scheduler.py`, line 22

### ❌ Current (Incorrect) Code
```python
def get_lr(self):
    t = self.last_epoch
    factor = self.lr_lambda(t)
    return [base_lr + factor for base_lr in self.base_lrs]  # WRONG: addition!
```

### 🐛 The Problem
- Uses addition instead of multiplication to apply the lambda factor
- Docstring says: `lr_t = base_lr * lr_lambda(t)`, but code does addition
- Example with `base_lr = 0.001`, `lr_lambda(t) = 0.5`:
  - **Wrong**: `0.001 + 0.5 = 0.501` (learning rate increases by 500×!)
  - **Correct**: `0.001 * 0.5 = 0.0005` (proper learning rate decay)
- Learning rate scaling is completely inverted

### ✅ Correct Formula (Lambda Scheduler)
$$\text{lr}_t = \text{base\_lr} \cdot f(t)$$

where $f(t)$ is the user-supplied lambda function returning a multiplicative factor.

### 🔧 Fix
Replace addition with multiplication:
```python
return [base_lr * factor for base_lr in self.base_lrs]  # Use *, not +
```

---


## Error 10: Adam Bias Correction — Multiplication Instead of Exponentiation

### 📍 Location
`Optimizers/adam.py`, lines 70–71

### ❌ Current (Incorrect) Code
```python
# Bias correction
bias_correction1 = 1.0 - beta1 * t        # WRONG: multiplication
bias_correction2 = 1.0 - beta2 * t        # WRONG: multiplication
m_hat = m / bias_correction1
v_hat = v / bias_correction2
```

### 🐛 The Problem
- Uses multiplication (`*`) instead of exponentiation (`**`)
- The docstring shows: `m_hat = m / (1 - beta1^t)` → exponentiation
- But code does: `1 - beta1 * t` → multiplication

Example with beta1 = 0.9 (default: 0.9):
- **At t=1**: 
  - Wrong: `1 - 0.9 * 1 = 0.1`
  - Correct: `1 - 0.9^1 = 0.1` (same)
  
- **At t=2**: 
  - Wrong: `1 - 0.9 * 2 = -0.8` ← **NEGATIVE!** 🚨
  - Correct: `1 - 0.9^2 = 0.19`

- **At t=10**: 
  - Wrong: `1 - 0.9 * 10 = -8.0` 
  - Correct: `1 - 0.9^10 ≈ 0.65`

**Consequence**: Dividing by negative bias correction flips the sign of parameter updates!

### ✅ Correct Formula (Adam Bias Correction)
$$m\_\text{hat} = \frac{m}{1 - \beta_1^t}$$

$$v\_\text{hat} = \frac{v}{1 - \beta_2^t}$$

### 🔧 Fix
```python
# Bias correction
bias_correction1 = 1.0 - beta1 ** t       # Use **, not *
bias_correction2 = 1.0 - beta2 ** t       # Use **, not *
m_hat = m / bias_correction1
v_hat = v / bias_correction2
```

---


## Error 11: Adam Second Moment — Missing Gradient Squaring

### 📍 Location
`Optimizers/adam.py`, line 73

### ❌ Current (Incorrect) Code
```python
# Update biased moment estimates
m.mul_(beta1).add_(grad, alpha=1.0 - beta1)      # Correct
v.mul_(beta2).add_(grad, alpha=1.0 - beta2)      # WRONG: missing ** 2
```

### 🐛 The Problem
- The docstring says: `v = beta2 * v + (1 - beta2) * grad^2`
- But the code applies the update to raw `grad`, not `grad ** 2`
- The second moment should track **variance** (squared gradients), not raw gradients

### 🔧 Fix
Add `** 2` to square the gradient:
```python
# Update biased moment estimates
m.mul_(beta1).add_(grad, alpha=1.0 - beta1)           # ✓ First moment
v.mul_(beta2).add_(grad ** 2, alpha=1.0 - beta2)      # ✅ Second moment (squared)
```

### 💥 Impact
- Adaptive learning rates don't adapt properly
- The term `1 / sqrt(v)` becomes meaningless
- Optimizer loses its core advantage
- Equivalent to non-adaptive gradient descent

---

## Error 12: SGDMomentum Optimizer — Velocity Update Uses Subtraction Instead of Addition

**File**: `Optimizers/sgd_momentum.py`, line 49  
**Severity**: HIGH  
**Impact**: Optimizer does gradient ascent instead of descent

### The Problem

The momentum accumulator should **add** gradients to build up velocity in the direction of steepest descent:
$$v = \mu \cdot v + \text{grad}$$

But the implementation **subtracts** instead:

```python
# WRONG ❌
v.mul_(mu).sub_(grad)      # Subtracts the gradient!
```

### Why This Breaks Everything

| Step | Gradient | Velocity (Wrong) | Update | Direction |
|------|----------|-----------------|--------|-----------|
| 1 | +0.1 | 0 - 0.1 = **-0.1** | p += (-lr) × (-0.1) = **+0.001** | 🔴 **UP (Ascending!)** |
| 2 | +0.1 | -0.09 - 0.1 = -0.19 | p += (-lr) × (-0.19) = **+0.0019** | 🔴 **UP (Still ascending!)** |
| 3 | +0.1 | -0.271 - 0.1 = -0.371 | p += (-lr) × (-0.371) = **+0.0037** | 🔴 **UP (Keeps ascending!)** |

The optimizer does **gradient ascent** instead of descent — the loss increases instead of decreases!

### The Fix

Change one line:

```python
# CORRECT ✅
v.mul_(mu).add_(grad)      # Add the gradient to accumulate velocity
```

Now velocity properly accumulates:

| Step | Gradient | Velocity (Correct) | Update | Direction |
|------|----------|-------------------|--------|-----------|
| 1 | +0.1 | 0 + 0.1 = **+0.1** | p += (-lr) × 0.1 = **-0.001** | 🟢 **DOWN (Descending!)** |
| 2 | +0.1 | +0.09 + 0.1 = +0.19 | p += (-lr) × 0.19 = **-0.0019** | 🟢 **DOWN (Accelerating down!)** |
| 3 | +0.1 | +0.271 + 0.1 = +0.371 | p += (-lr) × 0.371 = **-0.0037** | 🟢 **DOWN (Faster descent!)** |

With momentum, descent accelerates over time — exactly what we want.

### Mathematical Formula

**Correct momentum update**:
$$v_t = \mu \cdot v_{t-1} + \nabla L(p)$$
$$p_t = p_{t-1} - \text{lr} \cdot v_t$$

The accumulation of gradients creates **momentum** — like a rolling ball that speeds up downhill.

---

## Error 13: GroupNorm — Incorrect Reshape Dimension Order

**File**: `Models/Normalizations/groupnorm.py`, line 34  
**Severity**: HIGH  
**Impact**: Normalization computed over wrong channels; groups don't normalize independently

### The Problem

GroupNorm should normalize **within each group independently** over channels and spatial dimensions. But the reshape dimensions are swapped!

**Theory**: For input `[B, C, H, W]` with G groups and C//G channels per group:
1. ✅ Correct: Reshape to `[B, G, C//G, H, W]` then normalize over `[C//G, H, W]`
2. ❌ Wrong: Reshape to `[B, C//G, G, H, W]` then normalize over `[G, H, W]`

### Code Comparison

**Current Wrong**:
```python
x = x.view(B, C // self.G, self.G, *spatial)  # [B, C//G, G, ...]
dims = tuple(range(2, x.ndim))  # dims = (2,3,4,...) = [G, H, W]
```

**Correct**:
```python
x = x.view(B, self.G, C // self.G, *spatial)  # [B, G, C//G, ...]
dims = tuple(range(2, x.ndim))  # dims = (2,3,4,...) = [C//G, H, W]
```

### Example with Numbers

Input: `[B=2, C=256, H=28, W=28]` with `G=8` groups → `256/8 = 32` channels per group

| Configuration | Reshape | Normalize Over | Per-Group Size |
|---|---|---|---|
| **Wrong** | [2, 32, 8, 28, 28] | [8, 28, 28] | 6,272 elements per group ❌ |
| **Correct** | [2, 8, 32, 28, 28] | [32, 28, 28] | 25,088 elements per group ✓ |

**The bug**: Wrong version normalizes over only 6,272 values per group instead of 25,088 — that's **4× too few values**!

### Why This Breaks Training

- Each group only sees 1/4 the channel information it should
- Mean/variance estimates become noisy and unstable
- Batch effects leak across groups (normalization doesn't isolate groups)
- Gradient flow becomes irregular
- Model's ability to learn group-wise representations is broken

---

## Error 14: Pointer Network — Incorrect Matmul Dimension Order

**File**: `Models/heads.py`, lines 25-26  
**Severity**: HIGH  
**Impact**: Shape mismatch error; forward pass crashes when computing span predictions

### The Problem

The Pointer network uses a bilinear dot product to compute span positions. But the weight vector and feature matrix aren't compatible for matrix multiplication!

**Current Wrong Code**:
```python
Y1 = torch.matmul(self.w1, X1)     # w1: [2C], X1: [B, 2C, L]
Y2 = torch.matmul(self.w2, X2)     # w2: [2C], X2: [B, 2C, L]
```

### Why This Fails

`torch.matmul([2C], [B, 2C, L])` tries to do matrix multiplication:
- Expands to: `[1, 2C] @ [B, 2C, L]`
- Inner dimensions must match: `2C` should equal the first dimension of the right matrix
- But the first dimension is `B` (batch size), not `2C`
- **Shape mismatch error** → RuntimeError

### The Fix

**Option 1** (Transpose):
```python
Y1 = torch.matmul(X1.transpose(1, 2), self.w1)  # [B, L, 2C] @ [2C] = [B, L] ✓
Y2 = torch.matmul(X2.transpose(1, 2), self.w2)
```

**Option 2** (Einsum — more explicit):
```python
Y1 = torch.einsum('c,bcl->bl', self.w1, X1)  # Dot product over channel dimension
Y2 = torch.einsum('c,bcl->bl', self.w2, X2)
```

### Dimension Walkthrough

| Operation | Shape |
|-----------|-------|
| w1 | [2C] = [192] |
| X1 | [B, 2C, L] = [2, 192, 400] |
| X1.transpose(1,2) | [B, L, 2C] = [2, 400, 192] |
| matmul result | [B, L] = [2, 400] ✓ |

### Impact

- **Immediate crash**: RuntimeError when forward pass reaches Pointer layer
- No span predictions computed
- Training loop can't proceed
- Model can't make any predictions on answer spans

---

## Summary: All 16 Errors at a Glance

| # | Component | Current | Correct | Impact | Severity |
|---|-----------|---------|---------|--------|----------|
| 1 | Xavier Uniform | `/ (fan_in * fan_out)` | `/ (fan_in + fan_out)` | Weights too small | HIGH |
| 2 | Kaiming Uniform | `sqrt(1.0 / fan)` | `sqrt(2.0 / fan)` | Dead ReLU | HIGH |
| 3 | StepLR | `base_lr * gamma * (t // step)` | `base_lr * (gamma ** (t // step))` | LR=0 | **CRITICAL** |
| 4 | Cosine Scheduler | `(1 + cos(...))` | `0.5 * (1 + cos(...))` | Wrong schedule | MEDIUM |
| 5 | Adam Weight Decay | `alpha=-wd` | `alpha=wd` | Opposite effect | HIGH |
| 6 | Multi-Head Attention | No scaling | `* (1 / sqrt(d_k))` | Unstable gradients | HIGH |
| 7 | EncoderBlock | `out = res` (overwrite) + duplicate conv | `out = out + res` + single conv | Broken architecture | **CRITICAL** |
| 8 | Early Stopping | AND logic | OR logic | May train too long | LOW |
| 9 | Lambda Scheduler | `base_lr + factor` | `base_lr * factor` | Wrong LR scaling | HIGH |
| 10 | Adam Bias Correction | `1 - beta * t` | `1 - beta ** t` | Negative correction | HIGH |
| 11 | Adam Second Moment | no `** 2` on grad | `grad ** 2` | No variance tracking | HIGH |
| 12 | SGDMomentum Velocity | `.sub_(grad)` | `.add_(grad)` | Gradient ascent | HIGH |
| 13 | GroupNorm Reshape | `[B, C//G, G]` | `[B, G, C//G]` | Wrong normalization | HIGH |
| 14 | Pointer Matmul | `matmul(w, X)` wrong dims | `einsum('c,bcl->bl',w,X)` | Shape mismatch crash | HIGH |
| 15 | Lambda Scheduler Pickling | `lambda _: 1.0` | Named function | Can't save checkpoints | HIGH |
| 16 | Lambda Scheduler Warmup | Adam `lr=1.0` + no warmup | Adam `lr=args.learning_rate` + warmup | LR=1.0 too high | HIGH |

---

## Summary: All 16 Errors at a Glance

| # | Component | Current | Correct | Impact | Severity |
|---|-----------|---------|---------|--------|----------|
| 1 | Xavier Uniform | `/ (fan_in * fan_out)` | `/ (fan_in + fan_out)` | Weights too small | HIGH |
| 2 | Kaiming Uniform | `sqrt(1.0 / fan)` | `sqrt(2.0 / fan)` | Dead ReLU | HIGH |
| 3 | StepLR | `base_lr * gamma * (t // step)` | `base_lr * (gamma ** (t // step))` | LR=0 | **CRITICAL** |
| 4 | Cosine Scheduler | `(1 + cos(...))` | `0.5 * (1 + cos(...))` | Wrong schedule | MEDIUM |
| 5 | Adam Weight Decay | `alpha=-wd` | `alpha=wd` | Opposite effect | HIGH |
| 6 | Multi-Head Attention | No scaling | `* (1 / sqrt(d_k))` | Unstable gradients | HIGH |
| 7 | EncoderBlock | `out = res` (overwrite) + duplicate conv | `out = out + res` + single conv | Broken architecture | **CRITICAL** |
| 8 | Early Stopping | AND logic | OR logic | May train too long | LOW |
| 9 | Lambda Scheduler | `base_lr + factor` | `base_lr * factor` | Wrong LR scaling | HIGH |
| 10 | Adam Bias Correction | `1 - beta * t` | `1 - beta ** t` | Negative correction | HIGH |
| 11 | Adam Second Moment | no `** 2` on grad | `grad ** 2` | No variance tracking | HIGH |
| 12 | SGDMomentum Velocity | `.sub_(grad)` | `.add_(grad)` | Gradient ascent | HIGH |
| 13 | GroupNorm Reshape | `[B, C//G, G]` | `[B, G, C//G]` | Wrong normalization | HIGH |
| 14 | Pointer Matmul | `matmul(w, X)` wrong dims | `einsum('c,bcl->bl',w,X)` | Shape mismatch crash | HIGH |
| 15 | Lambda Scheduler Pickling | `lambda _: 1.0` | Named function | Can't save checkpoints | HIGH |
| 16 | Lambda Scheduler Warmup | Adam `lr=1.0` + no warmup | Adam `lr=args.learning_rate` + warmup | LR=1.0 too high | HIGH |

---

## Error 15: Lambda Scheduler — Unpicklable Lambda Function

**Location**: `Schedulers/scheduler.py`, line 24

**Severity**: 🔴 HIGH

### The Problem

Anonymous lambda functions cannot be serialized (pickled) by PyTorch. When training saves a checkpoint, the scheduler must be pickled with all its components, including the `lr_lambda` function. Anonymous lambdas have no module location and cannot be pickled.

### Current Wrong Code

```python
def lambda_scheduler(optimizer, args):
    return LambdaLR(optimizer, lr_lambda=lambda _: 1.0)  # ❌ Anonymous lambda
```

### What Happens

When you try to save a checkpoint after the first validation:

```
AttributeError: Can't get local object 'lambda_scheduler.<locals>.<lambda>'
```

This forces users to **only use "cosine" or "step" schedulers**, preventing use of "lambda" entirely.

### Correct Code

```python
def constant_lambda(t):
    """Constant learning rate (no decay)."""
    return 1.0

def lambda_scheduler(optimizer, args):
    return LambdaLR(optimizer, lr_lambda=constant_lambda)  # ✅ Named function is picklable
```

### Why This Works

- Named functions have an addressable location in the module
- PyTorch can serialize them during checkpoint saves
- The scheduler can be used normally without AttributeError

### Impact

- **Immediate crash**: AttributeError when trying to save checkpoint after first validation
- Cannot use "lambda" scheduler option, must use "cosine" or "step" instead
- Blocks custom learning rate schedules that users might want to define

---

## Error 16: Lambda Scheduler + Adam — Missing Warmup & Wrong Base LR

**Location**: `Optimizers/optimizer.py` (line 14) AND `Schedulers/scheduler.py` (lines 25-31)

**Severity**: 🔴 HIGH

### The Problem

This is a coordination issue between optimizer and scheduler. Two misaligned configurations:
1. Adam was hardcoded to `lr=1.0` (ignoring `args.learning_rate`)
2. Lambda scheduler only returned constant 1.0 (no actual warmup)
3. Result: Final LR = 1.0 × 1.0 = **1.0** ❌ WAY too high

### Current Wrong Code

```python
# Optimizers/optimizer.py
def adam(params, args):
    return Adam(
        params=params,
        lr=1.0,  # ❌ Hardcoded; ignores args.learning_rate
        betas=(args.beta1, args.beta2),
        eps=getattr(args, "eps", 1e-7),
        weight_decay=args.weight_decay,
    )

# Schedulers/scheduler.py
def constant_lambda(t):
    return 1.0  # ❌ Always constant, no warmup
```

### Correct Code

```python
# Optimizers/optimizer.py
def adam(params, args):
    return Adam(
        params=params,
        lr=args.learning_rate,  # ✅ Use actual learning rate
        betas=(args.beta1, args.beta2),
        eps=getattr(args, "eps", 1e-7),
        weight_decay=args.weight_decay,
    )

# Schedulers/scheduler.py
def warmup_lambda(t):
    """Inverse exponential warmup from QANet paper."""
    if t < 1000:
        tau = 250.0
        return 1.0 - math.exp(-t / tau)  # Warmup factor: 0→1
    else:
        return 1.0  # Constant afterwards
```

### Correct Behavior

With `learning_rate=0.001`:

| Step | Warmup Factor | Final LR |
|------|---|---|
| 0 | 0.0 | ~0.0 |
| 500 | 0.865 | 0.000865 |
| 1000 | 0.982 | 0.000982 |
| 5000+ | 1.0 | 0.001 |

### Impact

- **Without fix**: Final LR = 1.0 (catastrophic; gradient updates way too large)
- **With fix**: Final LR follows QANet paper spec (0→0.001 over 1000 steps)
- **Violates paper**: QANet paper explicitly requires this warmup scheme
- **Training fails**: With wrong LR, model immediately diverges or oscillates

### Notes

- This error only manifests when using `optimizer="adam"` + `scheduler="lambda"`
- Other combinations (adam+cosine, sgd+cosine, etc.) are unaffected
- Requires coordinated fix in BOTH files

---

## Priority Action Plan

### Tier 1: CRITICAL (Fix Immediately)
1. **Error 3**: StepLR scheduler → learning rate becomes zero
2. **Error 7**: EncoderBlock residual → self-attention is discarded

### Tier 2: HIGH (Fix Next)
3. **Error 1**: Xavier initialization
4. **Error 2**: Kaiming initialization
5. **Error 5**: Adam weight decay sign
6. **Error 6**: Attention scaling factor
7. **Error 9**: Lambda scheduler addition→multiplication
8. **Error 10**: Adam bias correction exponentiation
9. **Error 11**: Adam second moment gradient squaring
10. **Error 12**: SGDMomentum velocity subtraction→addition
11. **Error 13**: GroupNorm reshape dimension order
12. **Error 14**: Pointer matmul dimension transpose
13. **Error 15**: Lambda scheduler pickling → use named function
14. **Error 16**: Lambda scheduler warmup → implement QANet warmup + use args.learning_rate in Adam

### Tier 3: MEDIUM
15. **Error 4**: Cosine scheduler 0.5 factor

### Tier 4: LOW (Optional)
16. **Error 8**: Early stopping logic

---

## Validation Checklist

After applying all fixes, verify:

✅ **Training Loss**
- [ ] Loss decreases monotonically (or mostly monotonically) in early epochs
- [ ] No NaN or Inf values
- [ ] Stable numerical behavior

✅ **Gradients**
- [ ] Gradient norm changes naturally over training
- [ ] Not all zeros (would indicate dead gradients)
- [ ] Gradient clipping is triggered occasionally (not always)

✅ **Learning Rate**
- [ ] Starts at specified learning rate
- [ ] Decays smoothly over time
- [ ] Never becomes zero (unless using cosine to eta_min)

✅ **Metrics**
- [ ] F1 and EM improve over time
- [ ] Dev metrics improve faster than random (not just noise)
- [ ] Attention weights are well-distributed (not all on one token)

✅ **Model Architecture**
- [ ] Forward pass completes without errors
- [ ] Backward pass computes gradients for all parameters
- [ ] No shape mismatches or dimension errors

---